# Customer churn prediction tutorial
Run the training, comparison, calibration, and freeze commands in the project README first. This worked example uses the verified training release, frozen artifact, and the custodian's published aggregate evaluation.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.train_data import load_train
from src.cross_validate import load_folds, load_oof
from src.compare_models import load_selected_model
from src.data_contract import FEATURE_COLUMNS, TARGET_COLUMN
from src.preprocessing import churn_probabilities
from src.artifacts import read_json

training = load_train(root / 'data')
folds = load_folds(training, root / 'data' / 'folds.csv')
results_dir = root / 'results'
selected = load_selected_model(training, results_dir / 'selected_model.json')
display(folds.groupby('fold').size().rename('validation_customers').to_frame())

## Development scores
The comparison uses eight fixed configurations. The one-standard-error rule selects the simplest eligible configuration by mean fold average precision. Fit that configuration on four training folds and verify its scores on the remaining training fold.

In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score

stored = load_oof(training, folds, selected, results_dir)
valid_ids = set(folds.loc[folds['fold'].eq(0), 'row_id'])
validation = training['row_id'].isin(valid_ids)
features = training.loc[:, list(FEATURE_COLUMNS)]
labels = training[TARGET_COLUMN]
model = selected.build().fit(features.loc[~validation], labels.loc[~validation])
p_churn = churn_probabilities(model, features.loc[validation])
np.testing.assert_allclose(p_churn, stored.loc[stored['fold'].eq(0), 'p_churn'], rtol=1e-7, atol=1e-8)
print(f'{selected.key} fold 0 AP: {average_precision_score(labels.loc[validation], p_churn):.4f}')
comparison = pd.read_csv(results_dir / 'model_comparison.csv')
display(comparison[['configuration', 'mean_ap', 'sd_ap', 'mean_roc_auc', 'selected']])

## Calibrated probability and policy
Nested training folds compare the baseline and calibrated probabilities on matching customers. The following example uses a fictional customer and applies the same full-precision decision function as the app.

In [ ]:
from src.app import format_decision, score_customer
from src.freeze_model import load_frozen_model

calibration = read_json(results_dir / 'calibration' / 'report.json')
display(pd.Series(calibration['mean_paired_metrics'], name='calibrated minus baseline'))
frozen = load_frozen_model(root / 'models' / 'freeze_manifest.json')
fictional_customer = {
    'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'No', 'Dependents': 'No',
    'tenure': 12, 'PhoneService': 'Yes', 'MultipleLines': 'No', 'InternetService': 'DSL',
    'OnlineSecurity': 'Yes', 'OnlineBackup': 'No', 'DeviceProtection': 'No', 'TechSupport': 'No',
    'StreamingTV': 'No', 'StreamingMovies': 'No', 'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes', 'PaymentMethod': 'Electronic check', 'MonthlyCharges': 55.0,
    'TotalCharges': None,
}
probability, contact = score_customer(frozen, fictional_customer)
rule, detail = format_decision(probability, frozen.policy)
print(f'Fictional customer probability: {probability:.1%}')
print('Contact' if contact else 'Do not contact', '|', rule)
if detail:
    print(detail)

## Independent snapshot evaluation
This cell reads the custodian's aggregate JSON. Its results describe the frozen snapshot holdout. Future forecasting accuracy and actual financial impact remain open questions.

In [ ]:
test_path = results_dir / 'test_metrics.json'
if not test_path.is_file():
    print('The custodian evaluation is pending. Training checks are complete.')
else:
    test = read_json(test_path)
    if test['freeze_id'] != frozen.decision['freeze_id'] or test['policy'] != frozen.decision['policy']:
        raise ValueError('Published metrics do not match the loaded freeze and policy.')
    metrics = ('average_precision', 'roc_auc', 'brier', 'precision', 'recall', 'f1')
    intervals = test['bootstrap']['intervals']
    display(pd.DataFrame([{'metric': name, 'estimate': test[name], **intervals[name]} for name in metrics]))
    print(f"Holdout customers: {test['customers']:,}; positives: {test['positives']:,}; contacted: {test['contacted']:,}")
    display(pd.DataFrame(test['scenario_costs']).T)
    display(pd.DataFrame(test['paired_savings']).T)